In [ ]:
import pandas as pd
from pathlib import Path

# --- File paths (update these as needed) ---
data_dir = Path('/content')  # base directory for input files
output_dir = Path('/content')  # directory to save output

touchable_file = data_dir / 'touchable_resale_final_round.csv'
unique_file = data_dir / 'unique_to_dataset1.csv'
parsed_file = data_dir / 'ParsedResale_updatedSept2025.csv'
duplicates_output_file = output_dir / 'duplicate_property_keys_review.csv'
combined_output_file = output_dir / 'touchable_resale_final_round_combined_reconciled.csv'

# --- Load datasets ---
df_touchable = pd.read_csv(touchable_file)
df_unique = pd.read_csv(unique_file)
df_parsed = pd.read_csv(parsed_file)

# --- Normalize column names ---
for df in [df_touchable, df_unique, df_parsed]:
    df.columns = df.columns.str.strip().str.title()

# --- Define key columns ---
key_cols = ['Town', 'Development', 'Address', 'Unit Number']

# --- Standardize and clean key columns ---
for df in [df_touchable, df_unique, df_parsed]:
    for col in key_cols:
        df[col] = (
            df[col]
            .astype(str)
            .fillna('')
            .str.strip()
            .str.lower()
            .str.replace('–', '-', regex=False)  # replace en dash with hyphen
        )

# --- Helper function to create property key ---
def make_key(df):
    return df[key_cols].agg('|'.join, axis=1)

# --- Create _property_key in all dataframes ---
for df in [df_touchable, df_unique, df_parsed]:
    df['_property_key'] = make_key(df)

# --- Combine touchable + unique ---
combined = pd.concat([df_touchable, df_unique], ignore_index=True)

# --- Inspect duplicates BEFORE dropping them ---
dupes = combined[combined.duplicated(subset=['_property_key'], keep=False)].copy()
dupes = dupes.sort_values('_property_key')
print(f"🔍 Found {dupes['_property_key'].nunique()} duplicate property keys ({len(dupes)} total rows).")

# Optional: view basic details of duplicates
display(dupes[['_property_key'] + key_cols])

# Optional: export duplicates for offline review
dupes.to_csv(duplicates_output_file, index=False)
print(f"📂 Duplicates exported to {duplicates_output_file}")

# --- Drop duplicate property keys (keep first occurrence) ---
combined = combined.drop_duplicates(subset=['_property_key'])

# --- Find any rows still missing compared to ParsedResale ---
missing_rows = df_parsed.loc[~df_parsed['_property_key'].isin(combined['_property_key'])].copy()
print(f"⚠️ Missing rows still not merged: {len(missing_rows)}")

# --- Append missing rows from ParsedResale ---
if len(missing_rows) > 0:
    combined = pd.concat([combined, missing_rows], ignore_index=True)
    combined = combined.drop_duplicates(subset=['_property_key'])
    print("✅ Appended missing rows from ParsedResale.")

# --- Save the reconciled dataset ---
combined.to_csv(combined_output_file, index=False)
print(f"\n✅ Final combined dataset saved as: {combined_output_file}")

# --- Summary statistics ---
print(f"Rows in ParsedResale: {len(df_parsed)}")
print(f"Rows in Touchable original: {len(df_touchable)}")
print(f"Rows added from Unique: {len(df_unique)}")
print(f"Final combined rows: {len(combined)}")
print(f"Row count difference: {len(df_parsed) - len(combined)}")


🔍 Found 7 duplicate property keys (14 total rows).


,_property_key,Town,Development,Address,Unit Number
3,bedford|the village at bedford woods|1301 albi...,bedford,the village at bedford woods,1301 albion road,1301
79,bedford|the village at bedford woods|1301 albi...,bedford,the village at bedford woods,1301 albion road,1301
53,hudson|esplanade at main street|250 main stree...,hudson,esplanade at main street,250 main street,211
144,hudson|esplanade at main street|250 main stree...,hudson,esplanade at main street,250 main street,211
127,hudson|esplanade at main street|250 main stree...,hudson,esplanade at main street,250 main street,317
155,hudson|esplanade at main street|250 main stree...,hudson,esplanade at main street,250 main street,317
30,marblehead|marblehead highlands|33 intrepid ci...,marblehead,marblehead highlands,33 intrepid cir.,305
102,marblehead|marblehead highlands|33 intrepid ci...,marblehead,marblehead highlands,33 intrepid cir.,305
78,north reading|central place|63 central street|101,north reading,central place,63 central street,101
81,north reading|central place|63 central street|101,north reading,central place,63 central street,101


📂 Duplicates exported to /content/duplicate_property_keys_review.csv
⚠️ Missing rows still not merged: 0

✅ Final combined dataset saved as: /content/touchable_resale_final_round_combined_reconciled.csv
Rows in ParsedResale: 166
Rows in Touchable original: 158
Rows added from Unique: 9
Final combined rows: 160
Row count difference: 6
